# Interpretable Cardiac Risk Prediction
### A Deep Learning Approach with SHAP-based Clinical Explanations

**Dataset:** UCI Heart Disease (Cleveland)  
**Model:** Multi-Layer Perceptron (MLP)  
**Explainability:** SHAP (SHapley Additive exPlanations)

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score, roc_curve
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import shap

import os
os.makedirs('outputs', exist_ok=True)

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)
print('SHAP version:', shap.__version__)

TensorFlow version: 2.21.0
SHAP version: 0.51.0


## 2. Load Dataset

In [2]:
df = pd.read_csv('heart.csv')

# Rename columns to human-readable names
column_names = {
    'age':      'Age',
    'sex':      'Sex',
    'cp':       'ChestPainType',
    'trestbps': 'RestingBP',
    'chol':     'Cholesterol',
    'fbs':      'FastingBS',
    'restecg':  'RestingECG',
    'thalach':  'MaxHR',
    'exang':    'ExerciseAngina',
    'oldpeak':  'Oldpeak',
    'slope':    'ST_Slope',
    'ca':       'MajorVessels',
    'thal':     'Thalassemia',
    'target':   'Target'
}
df.rename(columns=column_names, inplace=True)

print('Shape:', df.shape)
df.head()

Shape: (303, 14)


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,MajorVessels,Thalassemia,Target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


## 3. Data Preprocessing

In [3]:
# Missing values
print('Missing values per column:')
print(df.isnull().sum())
print()

# Duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f'Removed {duplicates} duplicate rows. New shape: {df.shape}')

# Data types
print()
print('Data types:')
print(df.dtypes)

Missing values per column:
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
MajorVessels      0
Thalassemia       0
Target            0
dtype: int64

Duplicate rows: 1
Removed 1 duplicate rows. New shape: (302, 14)

Data types:
Age                 int64
Sex                 int64
ChestPainType       int64
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG          int64
MaxHR               int64
ExerciseAngina      int64
Oldpeak           float64
ST_Slope            int64
MajorVessels        int64
Thalassemia         int64
Target              int64
dtype: object


In [4]:
# Basic statistics
df.describe()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,MajorVessels,Thalassemia,Target
count,302.00000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,54.42053,0.682119,0.963576,131.602649,246.500000,0.149007,0.526490,149.569536,0.327815,1.043046,1.397351,0.718543,2.314570,0.543046
std,9.04797,0.466426,1.032044,17.563394,51.753489,0.356686,0.526027,22.903527,0.470196,1.161452,0.616274,1.006748,0.613026,0.498970
min,29.00000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.00000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.250000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.50000,1.000000,1.000000,130.000000,240.500000,0.000000,1.000000,152.500000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.00000,1.000000,2.000000,140.000000,274.750000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.00000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [5]:
# Separate features and target
X = df.drop('Target', axis=1)
y = df['Target']

feature_names = X.columns.tolist()

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples : {X_train_scaled.shape[0]}')
print(f'Testing samples  : {X_test_scaled.shape[0]}')
print(f'Features         : {X_train_scaled.shape[1]}')

Training samples : 241
Testing samples  : 61
Features         : 13


## 4. Exploratory Data Analysis

In [6]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 4))
counts = y.value_counts().sort_index()
ax.bar(['No Risk (0)', 'Cardiac Risk (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150)
plt.show()
print(counts)

Target
0    138
1    164
Name: count, dtype: int64


In [7]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, ax=ax, linewidths=0.5
)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', dpi=150)
plt.show()

In [8]:
# Distribution of key clinical features by target
clinical_features = ['Age', 'Cholesterol', 'MaxHR', 'RestingBP', 'Oldpeak']

fig, axes = plt.subplots(1, len(clinical_features), figsize=(18, 4))
for ax, feat in zip(axes, clinical_features):
    df[df['Target'] == 0][feat].plot(kind='hist', ax=ax, alpha=0.6, color='steelblue', label='No Risk', bins=20)
    df[df['Target'] == 1][feat].plot(kind='hist', ax=ax, alpha=0.6, color='tomato',    label='Risk',    bins=20)
    ax.set_title(feat)
    ax.set_xlabel('')
    ax.legend(fontsize=7)
fig.suptitle('Feature Distribution by Cardiac Risk', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [9]:
# Box plots: clinical features vs target
fig, axes = plt.subplots(1, len(clinical_features), figsize=(18, 4))
for ax, feat in zip(axes, clinical_features):
    df.boxplot(column=feat, by='Target', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Target (0=No Risk, 1=Risk)')
plt.suptitle('Box Plots: Clinical Features vs Cardiac Risk', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Deep Learning Model (MLP)

In [10]:
def build_model(input_dim):
    model = Sequential([
        Dense(64,  activation='relu', input_shape=(input_dim,)),
        Dropout(0.3),
        Dense(32,  activation='relu'),
        Dropout(0.3),
        Dense(16,  activation='relu'),
        Dense(1,   activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model(X_train_scaled.shape[1])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,521 (13.75 KB)

 Trainable params: 3,521 (13.75 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    epochs=200,
    batch_size=16,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.7500 - loss: 0.6502

13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.5049 - loss: 0.7110 - val_accuracy: 0.3514 - val_loss: 0.7047


Epoch 2/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7500 - loss: 0.5971

...

 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8125 - loss: 0.5186

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8824 - loss: 0.2769 - val_accuracy: 0.8108 - val_loss: 0.4019


Epoch 32/200


 1/13 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.8750 - loss: 0.3213

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8873 - loss: 0.2968 - val_accuracy: 0.8108 - val_loss: 0.4033


In [12]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/training_history.png', dpi=150)
plt.show()

## 6. Model Evaluation

In [13]:
y_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 40)
print('        Model Evaluation Results')
print('=' * 40)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 40)

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


        Model Evaluation Results
  Accuracy  : 0.8033  (80.33%)
  Precision : 0.7692
  Recall    : 0.9091
  F1-Score  : 0.8333
  ROC-AUC   : 0.8799


In [14]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['No Risk', 'Cardiac Risk'],
    yticklabels=['No Risk', 'Cardiac Risk']
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Positives  (TP): {tp}')
print(f'True Negatives  (TN): {tn}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')

True Positives  (TP): 30


True Negatives  (TN): 19
False Positives (FP): 9
False Negatives (FN): 3


In [15]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic (ROC) Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('outputs/roc_curve.png', dpi=150)
plt.show()

## 7. Explainable AI — SHAP

In [16]:
# Use a background sample for the SHAP explainer
background = X_train_scaled[:100]
explainer  = shap.KernelExplainer(model.predict, background)

# Compute SHAP values for the test set (use a subset for speed)
X_test_sample = X_test_scaled[:50]
shap_values   = explainer.shap_values(X_test_sample, nsamples=100)

# KernelExplainer returns a list for single output — unwrap if needed
if isinstance(shap_values, list):
    shap_values = shap_values[0]

# Squeeze trailing dim: KernelExplainer may return shape (n, 13, 1) but summary_plot expects (n, 13)
shap_values = np.array(shap_values)
if shap_values.ndim == 3 and shap_values.shape[-1] == 1:
    shap_values = shap_values[:, :, 0]

print('SHAP values shape:', shap_values.shape)

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


  0%|          | 0/50 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step


...

 54/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

143/313 ━━━━━━━━━━━━━━━━━━━━ 0s 865us/step

230/313 ━━━━━━━━━━━━━━━━━━━━ 0s 814us/step

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 801us/step


SHAP values shape: (50, 13)


In [17]:
# Global explanation — SHAP summary plot (beeswarm)
plt.figure()
shap.summary_plot(
    shap_values, X_test_sample,
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Summary Plot — Global Feature Importance', fontsize=12, pad=14)
plt.tight_layout()
plt.savefig('outputs/shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [18]:
# Global explanation — SHAP bar plot (mean absolute SHAP value)
plt.figure()
shap.summary_plot(
    shap_values, X_test_sample,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontsize=12, pad=14)
plt.tight_layout()
plt.savefig('outputs/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [19]:
# Local explanation — SHAP waterfall plot for one patient
patient_idx = 0

expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = float(np.array(expected_value).flatten()[0])

shap_explanation = shap.Explanation(
    values       = shap_values[patient_idx].flatten(),
    base_values  = expected_value,
    data         = X_test_sample[patient_idx].flatten(),
    feature_names= feature_names
)

plt.figure()
shap.plots.waterfall(shap_explanation, show=False)
plt.title(f'SHAP Local Explanation — Patient {patient_idx + 1}', fontsize=11, pad=14)
plt.tight_layout()
plt.savefig('outputs/shap_local_explanation.png', dpi=150, bbox_inches='tight')
plt.show()

actual    = y_test.iloc[patient_idx]
predicted = y_pred[patient_idx]
prob      = y_prob[patient_idx]
print(f'Patient {patient_idx + 1}: Actual={actual}, Predicted={predicted}, Probability={prob:.4f}')

Patient 1: Actual=0, Predicted=0, Probability=0.1568


## 8. Prediction vs Explanation — Sample Patients

In [20]:
n_patients = 10

rows = []
for i in range(n_patients):
    actual    = int(y_test.iloc[i])
    predicted = int(y_pred[i])
    prob      = round(float(y_prob[i]), 4)

    sv = shap_values[i].flatten()
    top_idx = np.argsort(np.abs(sv))[::-1][:3]
    top_features = ', '.join([feature_names[j] for j in top_idx])

    rows.append({
        'Patient': i + 1,
        'Actual':     'Risk' if actual    == 1 else 'No Risk',
        'Predicted':  'Risk' if predicted == 1 else 'No Risk',
        'Probability': prob,
        'Correct':    'Yes' if actual == predicted else 'No',
        'Top SHAP Features': top_features
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

 Patient  Actual Predicted  Probability Correct                          Top SHAP Features
       1 No Risk   No Risk       0.1568     Yes          Thalassemia, MaxHR, ChestPainType
       2 No Risk   No Risk       0.1686     Yes   ChestPainType, Thalassemia, MajorVessels
       3 No Risk   No Risk       0.0072     Yes          Thalassemia, MaxHR, ChestPainType
       4 No Risk      Risk       0.8629      No       ChestPainType, Oldpeak, MajorVessels
       5 No Risk      Risk       0.7580      No ChestPainType, Thalassemia, ExerciseAngina
       6 No Risk   No Risk       0.0184     Yes             MaxHR, Oldpeak, ExerciseAngina
       7    Risk      Risk       0.9431     Yes            Sex, ChestPainType, Thalassemia
       8 No Risk   No Risk       0.1351     Yes                Oldpeak, ChestPainType, Sex
       9    Risk      Risk       0.9208     Yes                  MaxHR, Sex, ChestPainType
      10 No Risk      Risk       0.8657      No        ChestPainType, Thalassemia, Oldpeak

## 9. Summary

In [21]:
print('=' * 50)
print('           PROJECT SUMMARY')
print('=' * 50)
print(f'Dataset     : UCI Heart Disease (Cleveland)')
print(f'Samples     : {len(df)}')
print(f'Features    : {len(feature_names)}')
print(f'Model       : Multi-Layer Perceptron (MLP)')
print(f'XAI Method  : SHAP (KernelExplainer)')
print()
print('--- Evaluation Metrics ---')
print(f'Accuracy    : {acc:.4f}')
print(f'Precision   : {prec:.4f}')
print(f'Recall      : {rec:.4f}')
print(f'F1-Score    : {f1:.4f}')
print(f'ROC-AUC     : {auc:.4f}')
print()
print('--- Output Files ---')
for f in sorted(os.listdir('outputs')):
    print(f'  outputs/{f}')
print('=' * 50)

           PROJECT SUMMARY
Dataset     : UCI Heart Disease (Cleveland)
Samples     : 302
Features    : 13
Model       : Multi-Layer Perceptron (MLP)
XAI Method  : SHAP (KernelExplainer)

--- Evaluation Metrics ---
Accuracy    : 0.8033
Precision   : 0.7692
Recall      : 0.9091
F1-Score    : 0.8333
ROC-AUC     : 0.8799

--- Output Files ---
  outputs/boxplots.png
  outputs/class_distribution.png
  outputs/confusion_matrix.png
  outputs/correlation_heatmap.png
  outputs/feature_distributions.png
  outputs/roc_curve.png
  outputs/shap_feature_importance.png
  outputs/shap_local_explanation.png
  outputs/shap_summary_plot.png
  outputs/training_history.png
